# Data preprocessing step

Convert raw waterfalls into cropped grayscale waterfalls. Remove failed downloads, use the same image dimensions and normalize.

In [77]:
import cv2
from pathlib import Path
import numpy as np
import os
import glob
from tqdm import tqdm

rawdata = '/Users/tedvtorov/Desktop/data'
outdata = 'data'
with_signal = 'with_signal'
without_signal = 'without_signal'

In [78]:
os.makedirs(outdata, exist_ok=True)
os.makedirs(os.path.join(outdata, with_signal), exist_ok=True)
os.makedirs(os.path.join(outdata, without_signal), exist_ok=True)

In [79]:
def crop_waterfall_and_grayscale(img,
                   black_threshold=40,
                   min_horizontal_fraction=0.7):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    is_black = gray < black_threshold
    x_probe = w // 4
    y_top = None
    for y in range(h):
        if is_black[y, x_probe]:
            y_top = y
            break
    if y_top is None:
        raise ValueError("Top border not found.")

    x_left = x_probe
    while x_left > 0 and is_black[y_top, x_left - 1]:
        x_left -= 1

    x_right = x_probe
    while x_right < w - 1 and is_black[y_top, x_right + 1]:
        x_right += 1

    span_width = x_right - x_left + 1
    y_bottom = None
    for y in range(h - 1, -1, -1):
        frac_black = is_black[y, x_left:x_right + 1].mean()
        if frac_black >= min_horizontal_fraction:
            y_bottom = y
            break
    if y_bottom is None:
        raise ValueError("Bottom border not found.")

    y0 = max(y_top + 1, 0)
    y1 = min(y_bottom, h - 1)
    x0 = max(x_left + 1, 0)
    x1 = min(x_right, w - 1)

    # Collect some statistics while im here
    dimensions = (x1 - x0, y1 - y0)
    avg_brightness = gray.mean()

    cropped = gray[y0:y1, x0:x1]
    ch, cw = cropped.shape[:2]
    if ch >= 1500 and cw >= 600:
        center_y = ch // 2
        center_x = cw // 2
        half_h = 1500 // 2
        half_w = 600 // 2
        start_y = max(center_y - half_h, 0)
        end_y = start_y + 1500
        start_x = max(center_x - half_w, 0)
        end_x = start_x + 600
        cropped = cropped[start_y:end_y, start_x:end_x]
        return np.array(cropped), (dimensions, avg_brightness)
    return None

In [80]:
def remove_edges(img, edge_width=10):  # they contain no data anyway
    h, w = img.shape[:2]
    cropped = img[0:h, edge_width:w - edge_width]
    return cropped

In [ ]:
withsignalstats = []
for filepath in tqdm(glob.glob(os.path.join(rawdata, with_signal, '*.png'))):
    img = cv2.imread(filepath)
    if img is None:
        continue
    result = crop_waterfall_and_grayscale(img)
    if result is not None:
        cropped, stats = result
        final = remove_edges(cropped, edge_width=10)
        withsignalstats.append(stats)
        outpath = os.path.join(outdata, with_signal, os.path.basename(filepath))
        cv2.imwrite(outpath, final)

100%|██████████| 39977/39977 [24:33<00:00, 27.14it/s]


In [84]:
nosignalstats = []
for filepath in tqdm(glob.glob(os.path.join(rawdata, without_signal, '*.png'))):
    img = cv2.imread(filepath)
    if img is None:
        continue
    result = crop_waterfall_and_grayscale(img)
    if result is not None:
        cropped, stats = result
        nosignalstats.append(stats)
        final = remove_edges(cropped, edge_width=10)
        outpath = os.path.join(outdata, without_signal, os.path.basename(filepath))
        cv2.imwrite(outpath, final)

100%|██████████| 39975/39975 [24:23<00:00, 27.31it/s]
